# Episode 24: Text to Cypher: our GPT asks a graph database

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep024-knowledge-graph.ipynb)

We fine-tune our agent to write Cypher queries for smollerGraph, the graph database built in our sibling series, and to answer from what comes back, then measure it with the graph and without.

[The video](https://youtu.be/NxeRk1Q8LXY) · [The episode's page](https://mvgpt.nenuadrian.com/ep024-knowledge-graph/) · [The docs](https://nenuadrian.github.io/mvGPT/knowledge-graph/) · [The code at `ep024/end`](https://github.com/nenuadrian/mvGPT/tree/ep024/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep024/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


import shlex
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def start(command, url):
    # A server in the background, once it answers at url; its output goes to /tmp/server-<port>.log.
    log = open(f"/tmp/server-{urllib.parse.urlparse(url).port}.log", "w")
    server = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT)
    while True:
        try:
            urllib.request.urlopen(url, timeout=1)
            return server
        except urllib.error.HTTPError:
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError(open(log.name).read())
            time.sleep(1)


def stop(server):
    server.terminate()
    server.wait()


fetch("agent")

## smollerGraph
The graph the model asks: smollerGraph as its episode 10 left it, loaded with the mammals and
birds on Wikidata. It needs CMake 3.28 or newer, newer than Colab's, and its configure
downloads abseil, GoogleTest and cpp-httplib.

In [ ]:
if not os.path.isdir("/content/smollerGraph"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep010/end https://github.com/nenuadrian/smollerGraph /content/smollerGraph
!pip install --quiet cmake ninja
!cmake -S /content/smollerGraph -B /content/smollerGraph/build -G Ninja > /dev/null && cmake --build /content/smollerGraph/build --target sg -- --quiet
!rm -f /content/smollerGraph/wikidata.sg && /content/smollerGraph/build/sg load /content/smollerGraph/wikidata.sg /content/smollerGraph/data/wikidata.txt

## The graph, over HTTP
`sg serve` in the background, on port 8474, where mvGPT looks for it, then the Red fox's
family, asked in Cypher.

In [ ]:
import json

sg = start("/content/smollerGraph/build/sg serve /content/smollerGraph/wikidata.sg 8474", "http://127.0.0.1:8474/api/names?prefix=R")
cypher = "MATCH (s:Species {name: 'Red fox'})-[:PARENT_TAXON*2]->(f:Family) RETURN f.name"
print(json.dumps(json.load(urllib.request.urlopen("http://127.0.0.1:8474/api/query?" + urllib.parse.urlencode({"q": cypher}))), indent=2))

## The tests
sg's answer as text, names in quotes, and a call in the loop, with the graph faked.

In [ ]:
!python -m pytest -q tests/test_graph.py

## The data
Thirty thousand questions, most of them about animals, each query asked of `sg serve` once,
into `data/graph/`.

In [ ]:
!python graph_data.py /content/smollerGraph/data/wikidata.txt

## Fine-tuning
Episode 21's agent, from the Hub, fine-tuned for twenty minutes, to `checkpoints/graph`. The
episode's run on a laptop took 1,024 steps.

In [ ]:
!python graph_sft.py 20

## Measuring it
The held-out sets through `sg serve`, then with the model writing sg's answer itself.

In [ ]:
!python graph_eval.py checkpoints/graph/model.pt

## Asking it
A species training never names, an animal the graph hasn't got, families training doesn't
ask about, a wording it never uses, and the Red fox with the graph off.

In [ ]:
!python graph.py checkpoints/graph/model.pt "What family is the Red fox in?"
!python graph.py checkpoints/graph/model.pt "How threatened is the King cobra?"
!python graph.py checkpoints/graph/model.pt "Which species in Manidae are critically endangered?"
!python graph.py checkpoints/graph/model.pt "Which species in Lemuridae are critically endangered?"
!python graph.py checkpoints/graph/model.pt "What do biologists call the Red fox?"
!python graph.py checkpoints/graph/model.pt "What family is the Red fox in?" --no-tool